# Tarea 1: Juego de la Vida de Conway
**Curso:** Computación Paralela  
**Universidad:** LEAD University  
**Profesor:** Johansell Villalobos Cubillo


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import time
import warnings

warnings.filterwarnings("ignore")


---
## 1. Implementación Orientada a Objetos — Clase `GameOfLife`


In [ ]:
class GameOfLife:
    """
    Implementación del Juego de la Vida de Conway.

    El tablero es una rejilla bidimensional de celdas binarias:
        1 = viva   |   0 = muerta

    Las reglas aplicadas en cada generación son:
        - Superpoblación : celda viva con >3 vecinos vivos  → muere
        - Soledad        : celda viva con <2 vecinos vivos  → muere
        - Supervivencia  : celda viva con 2 o 3 vecinos     → sobrevive
        - Reproducción   : celda muerta con exactamente 3   → nace
    """

    # ------------------------------------------------------------------
    def __init__(self, filas, cols, estado_inicial=None):
        """
        Parámetros
        ----------
        filas          : int  – número de filas de la rejilla
        cols           : int  – número de columnas de la rejilla
        estado_inicial : ndarray de forma (filas, cols) o None.
                         Si es None se genera un estado aleatorio.
        """
        self.filas = filas
        self.cols  = cols

        if estado_inicial is not None:
            self.tablero = np.array(estado_inicial, dtype=np.uint8)
        else:
            self.tablero = np.random.randint(0, 2, size=(filas, cols), dtype=np.uint8)

        self.generacion = 0

    # ------------------------------------------------------------------
    def _contar_vecinos(self):
        """
        Cuenta los vecinos vivos de cada celda usando desplazamientos
        de la rejilla (8-conectividad, bordes periódicos).

        Retorna
        -------
        vecinos : ndarray uint8 de forma (filas, cols)
        """
        t = self.tablero
        vecinos = (
            np.roll(np.roll(t, -1, axis=0), -1, axis=1) +  # ↗
            np.roll(t, -1, axis=0)                         +  # ↑
            np.roll(np.roll(t, -1, axis=0),  1, axis=1) +  # ↖
            np.roll(t,  1, axis=1)                         +  # ←
            np.roll(np.roll(t,  1, axis=0),  1, axis=1) +  # ↙
            np.roll(t,  1, axis=0)                         +  # ↓
            np.roll(np.roll(t,  1, axis=0), -1, axis=1) +  # ↘
            np.roll(t, -1, axis=1)                            # →
        )
        return vecinos

    # ------------------------------------------------------------------
    def step(self):
        """
        Avanza el tablero exactamente una generación aplicando las
        cuatro reglas de Conway de forma vectorizada.
        """
        vecinos = self._contar_vecinos()
        t = self.tablero

        # Reglas aplicadas simultáneamente
        nuevo = np.zeros_like(t)
        nuevo[(t == 1) & (vecinos == 2)] = 1   # Supervivencia
        nuevo[(t == 1) & (vecinos == 3)] = 1   # Supervivencia
        nuevo[(t == 0) & (vecinos == 3)] = 1   # Reproducción
        # Superpoblación y Soledad → quedan en 0 (ya inicializado)

        self.tablero = nuevo
        self.generacion += 1

    # ------------------------------------------------------------------
    def run(self, pasos):
        """
        Ejecuta múltiples iteraciones del juego.

        Parámetros
        ----------
        pasos : int – número de generaciones a avanzar
        """
        for _ in range(pasos):
            self.step()

    # ------------------------------------------------------------------
    def get_state(self):
        """
        Retorna una copia del estado actual del tablero.

        Retorna
        -------
        ndarray uint8 de forma (filas, cols)
        """
        return self.tablero.copy()

    # ------------------------------------------------------------------
    def contar_vivas(self):
        # Retorna el número de celdas vivas en la generación actual.
        return int(self.tablero.sum())

    # ------------------------------------------------------------------
    def __repr__(self):
        return (
            f"GameOfLife(filas={self.filas}, cols={self.cols}, "
            f"generación={self.generacion}, vivas={self.contar_vivas()})"
        )

print("Clase GameOfLife definida correctamente.")


---
## 2. Patrones Clásicos

Definimos los patrones iniciales más conocidos del Juego de la Vida.


In [ ]:
def crear_patron(filas, cols, nombre):
    """
    Crea una instancia de GameOfLife con un patrón clásico centrado.

    Patrones disponibles
    --------------------
    'aleatorio'  – estado aleatorio con ~30 % de celdas vivas
    'glider'     – nave espacial que se desplaza en diagonal
    'blinker'    – oscilador de periodo 2 (línea horizontal/vertical)
    'toad'       – oscilador de periodo 2 (dos filas escalonadas)
    'beacon'     – oscilador de periodo 2 (dos bloques en esquinas)
    'pulsar'     – oscilador de periodo 3 (patrón grande simétrico)
    'block'      – estructura estática (cuadrado 2×2)
    'beehive'    – estructura estática (panal de abeja)
    """
    tablero = np.zeros((filas, cols), dtype=np.uint8)
    cy, cx  = filas // 2, cols // 2   # centro del tablero

    if nombre == 'aleatorio':
        tablero = np.random.choice([0, 1], size=(filas, cols),
                                   p=[0.70, 0.30]).astype(np.uint8)

    elif nombre == 'glider':
        patron = np.array([[0, 1, 0],
                           [0, 0, 1],
                           [1, 1, 1]], dtype=np.uint8)
        tablero[cy-1:cy+2, cx-1:cx+2] = patron

    elif nombre == 'blinker':
        tablero[cy, cx-1:cx+2] = 1

    elif nombre == 'toad':
        tablero[cy,   cx-1:cx+2] = 1
        tablero[cy+1, cx-2:cx+1] = 1

    elif nombre == 'beacon':
        tablero[cy-1:cy+1, cx-1:cx+1] = 1
        tablero[cy+1:cy+3, cx+1:cx+3] = 1

    elif nombre == 'pulsar':
        # Coordenadas relativas del pulsar (patrón de periodo 3)
        filas_p = [-6,-5,-4, -1,1, 4,5,6]
        for dy in [-4,-3,-2,2,3,4]:
            for dx in filas_p:
                r, c = cy + dy, cx + dx
                if 0 <= r < filas and 0 <= c < cols:
                    tablero[r, c] = 1
        for dx in [-4,-3,-2,2,3,4]:
            for dy in filas_p:
                r, c = cy + dy, cx + dx
                if 0 <= r < filas and 0 <= c < cols:
                    tablero[r, c] = 1

    elif nombre == 'block':
        tablero[cy:cy+2, cx:cx+2] = 1

    elif nombre == 'beehive':
        patron = np.array([[0,1,1,0],
                           [1,0,0,1],
                           [0,1,1,0]], dtype=np.uint8)
        tablero[cy-1:cy+2, cx-2:cx+2] = patron

    else:
        raise ValueError(f"Patrón '{nombre}' no reconocido.")

    return GameOfLife(filas, cols, estado_inicial=tablero)


# Prueba rápida
juego = crear_patron(20, 20, 'glider')
print(juego)


---
## 3. Visualización

### 3.1 Visualización estática — evolución fotograma a fotograma


In [ ]:
def visualizar_evolucion(nombre_patron, filas=32, cols=32,
                         pasos=8, columnas_fig=4):
    """
    Muestra una cuadrícula de fotogramas con la evolución del patrón.

    Parámetros
    ----------
    nombre_patron : str  – nombre del patrón (ver crear_patron)
    filas / cols  : int  – tamaño de la rejilla
    pasos         : int  – número de generaciones a mostrar
    columnas_fig  : int  – columnas en la figura
    """
    juego  = crear_patron(filas, cols, nombre_patron)
    filas_fig = (pasos + columnas_fig - 1) // columnas_fig

    fig, ejes = plt.subplots(filas_fig, columnas_fig,
                             figsize=(columnas_fig * 2.5, filas_fig * 2.5))
    ejes = np.array(ejes).flatten()

    for i in range(pasos):
        ejes[i].imshow(juego.get_state(), cmap='binary',
                       interpolation='nearest', vmin=0, vmax=1)
        ejes[i].set_title(f"Gen {juego.generacion}", fontsize=9)
        ejes[i].axis('off')
        juego.step()

    for j in range(pasos, len(ejes)):
        ejes[j].axis('off')

    fig.suptitle(f"Evolución — Patrón: {nombre_patron.capitalize()}  "
                 f"({filas}×{cols})", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Mostrar varios patrones
for patron in ['glider', 'blinker', 'toad', 'beacon', 'pulsar']:
    visualizar_evolucion(patron, filas=32, cols=32, pasos=8)


### 3.2 Animación interactiva

In [ ]:
def animar_juego(nombre_patron, filas=64, cols=64,
                 generaciones=100, intervalo_ms=100):
    """
    Genera una animación del Juego de la Vida para un patrón dado.

    Parámetros
    ----------
    nombre_patron  : str – patrón inicial
    filas / cols   : int – tamaño de la rejilla
    generaciones   : int – fotogramas totales de la animación
    intervalo_ms   : int – tiempo entre fotogramas en milisegundos
    """
    juego = crear_patron(filas, cols, nombre_patron)

    fig, ax = plt.subplots(figsize=(6, 6))
    img = ax.imshow(juego.get_state(), cmap='inferno',
                    interpolation='nearest', vmin=0, vmax=1)
    ax.axis('off')
    titulo = ax.set_title(f"{nombre_patron.capitalize()} — Gen 0",
                          fontsize=12)

    def actualizar(frame):
        juego.step()
        img.set_data(juego.get_state())
        titulo.set_text(
            f"{nombre_patron.capitalize()} — Gen {juego.generacion} "
            f"| Vivas: {juego.contar_vivas()}"
        )
        return [img, titulo]

    ani = animation.FuncAnimation(
        fig, actualizar,
        frames=generaciones,
        interval=intervalo_ms,
        blit=True
    )

    plt.tight_layout()
    plt.show()
    return ani


ani_glider  = animar_juego('glider',   filas=64, cols=64, generaciones=80)
ani_random  = animar_juego('aleatorio', filas=64, cols=64, generaciones=80)


---
## 4. Medición de Rendimiento y Complejidad Empírica


In [ ]:
def medir_rendimiento(tamanos, repeticiones=5, pasos_por_prueba=10):
    """
    Mide el tiempo promedio por iteración para distintos tamaños de rejilla.

    Parámetros
    ----------
    tamanos          : list[int] – lista de dimensiones N (rejilla N×N)
    repeticiones     : int       – número de ejecuciones para promediar
    pasos_por_prueba : int       – pasos por cada ejecución

    Retorna
    -------
    resultados : dict con claves 'n_celdas', 'tiempo_medio', 'tiempo_std'
    """
    n_celdas    = []
    tiempo_medio = []
    tiempo_std  = []

    for N in tamanos:
        tiempos = []
        for _ in range(repeticiones):
            juego = crear_patron(N, N, 'aleatorio')
            inicio = time.perf_counter()
            juego.run(pasos_por_prueba)
            fin = time.perf_counter()
            tiempos.append((fin - inicio) / pasos_por_prueba)

        n_celdas.append(N * N)
        tiempo_medio.append(np.mean(tiempos))
        tiempo_std.append(np.std(tiempos))
        print(f"  N={N:5d} ({N}×{N}) | celdas={N*N:9,} | "
              f"t_media={np.mean(tiempos)*1e3:.3f} ms")

    return {
        'n_celdas'   : np.array(n_celdas),
        'tiempo_medio': np.array(tiempo_medio),
        'tiempo_std' : np.array(tiempo_std)
    }


TAMANOS = [32, 64, 128, 256, 512, 1024]

print("Ejecutando benchmark…")
resultados = medir_rendimiento(TAMANOS, repeticiones=5, pasos_por_prueba=10)
print("\nBenchmark completado.")


In [ ]:
def graficar_rendimiento(resultados):
    """
    Genera dos gráficas de rendimiento:
        1) Escala lineal con curvas teóricas de referencia
        2) Escala log-log para identificar la clase de complejidad

    Parámetros
    ----------
    resultados : dict – salida de medir_rendimiento()
    """
    n  = resultados['n_celdas']
    t  = resultados['tiempo_medio']
    sd = resultados['tiempo_std']

    # Curvas teóricas normalizadas al primer punto medido
    factor = t[0] / n[0]
    c_lineal    = factor * n
    c_nlogn     = factor * n * np.log2(n) / np.log2(n[0])
    c_cuadratica = factor * (n / n[0]) ** 2 * t[0]

    fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Gráfica 1: escala lineal ─────────────────────────────────────
    ax = ejes[0]
    ax.errorbar(n, t * 1e3, yerr=sd * 1e3,
                fmt='o-', color='steelblue', linewidth=2,
                markersize=7, capsize=4, label='Medido')
    ax.plot(n, c_lineal    * 1e3, '--', color='green',  label='O(n)')
    ax.plot(n, c_nlogn     * 1e3, '--', color='orange', label='O(n log n)')
    ax.plot(n, c_cuadratica* 1e3, '--', color='red',    label='O(n²)')
    ax.set_xlabel('Número de celdas  (N×N)', fontsize=11)
    ax.set_ylabel('Tiempo por iteración  (ms)', fontsize=11)
    ax.set_title('Rendimiento — Escala lineal', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.4)

    # ── Gráfica 2: escala log-log ────────────────────────────────────
    ax = ejes[1]
    ax.errorbar(n, t * 1e3, yerr=sd * 1e3,
                fmt='o-', color='steelblue', linewidth=2,
                markersize=7, capsize=4, label='Medido')
    ax.plot(n, c_lineal    * 1e3, '--', color='green',  label='O(n)')
    ax.plot(n, c_nlogn     * 1e3, '--', color='orange', label='O(n log n)')
    ax.plot(n, c_cuadratica* 1e3, '--', color='red',    label='O(n²)')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Número de celdas  (N×N)  [escala log]', fontsize=11)
    ax.set_ylabel('Tiempo por iteración  (ms)  [escala log]', fontsize=11)
    ax.set_title('Rendimiento — Escala log-log', fontsize=12)
    ax.legend()
    ax.grid(True, which='both', alpha=0.4)

    plt.suptitle('Complejidad Empírica — Juego de la Vida de Conway',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


graficar_rendimiento(resultados)


---
## 5. Análisis y Discusión


In [ ]:
def analizar_resultados(resultados):
    """
    Estima empíricamente el exponente de complejidad ajustando una
    regresión lineal en escala log-log   log(t) = α·log(n) + β.

    Un exponente α ≈ 1 indica complejidad lineal O(n).

    Parámetros
    ----------
    resultados : dict – salida de medir_rendimiento()
    """
    n = resultados['n_celdas']
    t = resultados['tiempo_medio']

    log_n = np.log10(n)
    log_t = np.log10(t)
    coefs = np.polyfit(log_n, log_t, 1)

    print("=" * 50)
    print("  Análisis de complejidad empírica")
    print("=" * 50)
    print(f"  Exponente estimado (α) : {coefs[0]:.4f}")
    print(f"  Intercepto (β)         : {coefs[1]:.4f}")
    print()
    print("  Interpretación:")
    if coefs[0] < 1.1:
        print("  → Complejidad ≈ O(n)  — crecimiento LINEAL")
        print("  → La implementación vectorizada con NumPy escala bien.")
    elif coefs[0] < 1.6:
        print("  → Complejidad ≈ O(n·log n)  — crecimiento casi lineal")
    else:
        print("  → Complejidad superior a O(n)  — posible cuello de botella")
    print()

    # Uso de memoria estimado
    print("  Uso de memoria estimada por rejilla:")
    for N in [32, 64, 128, 256, 512, 1024]:
        mem_kb = (N * N * 1) / 1024       # uint8 → 1 byte por celda
        mem_kb_doble = mem_kb * 2          # tablero actual + buffer
        print(f"    {N:5d}×{N:<5d}  →  {mem_kb_doble:.1f} KB")

    print()
    print("  Cuellos de botella observados:")
    print("  • np.roll() crea copias intermedias en memoria.")
    print("  • Para rejillas > 1024×1024 el GIL de Python y la cache L2/L3")
    print("    pueden degradar el rendimiento.")
    print("  • Alternativa: scipy.ndimage.convolve o Numba @jit para acelerar.")
    print("=" * 50)


analizar_resultados(resultados)


In [ ]:
def graficar_poblacion(nombre_patron='aleatorio', filas=64, cols=64,
                       generaciones=200):
    """
    Traza la cantidad de celdas vivas a lo largo de las generaciones,
    mostrando la dinámica de población del sistema.

    Parámetros
    ----------
    nombre_patron : str – patrón inicial
    filas / cols  : int – tamaño de la rejilla
    generaciones  : int – número de generaciones a simular
    """
    juego = crear_patron(filas, cols, nombre_patron)
    historial = [juego.contar_vivas()]

    for _ in range(generaciones):
        juego.step()
        historial.append(juego.contar_vivas())

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(historial, color='steelblue', linewidth=1.5)
    ax.fill_between(range(len(historial)), historial, alpha=0.2, color='steelblue')
    ax.set_xlabel('Generación', fontsize=11)
    ax.set_ylabel('Celdas vivas', fontsize=11)
    ax.set_title(
        f'Dinámica de población — {nombre_patron.capitalize()}  '
        f'({filas}×{cols})',
        fontsize=12
    )
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()


graficar_poblacion('aleatorio', filas=64, cols=64, generaciones=200)
graficar_poblacion('glider',    filas=64, cols=64, generaciones=200)


---
## 6. Conclusiones

- **Implementación vectorizada:** El uso de `numpy.roll` permite aplicar las reglas de Conway de forma completamente vectorizada, procesando toda la rejilla en una sola operación sin bucles explícitos en Python.

- **Complejidad empírica ≈ O(n):** La gráfica log-log confirma que el tiempo de ejecución crece de manera aproximadamente lineal con el número de celdas, lo que es el comportamiento esperado para operaciones vectorizadas sobre arreglos NumPy.

- **Memoria:** El consumo es proporcional al tamaño de la rejilla (1 byte/celda con `uint8`). Para 1024×1024 se requieren apenas ~2 MB, lo que hace la implementación viable en hardware modesto.

- **Cuello de botella principal:** `numpy.roll` genera copias intermedias del arreglo. Para rejillas muy grandes (> 2048×2048), una implementación con `scipy.ndimage.convolve` o aceleración con **Numba** puede reducir el tiempo entre 5× y 20×.

- **Comportamientos emergentes:** Patrones como el Glider y el Pulsar demuestran cómo reglas locales extremadamente simples generan estructuras globales complejas, estables y periódicas.
